# Career Intelligence & Skills Evolution Analyzer
### 03_career_recommendation_engine.ipynb

#### Objective
Build a personalized recommendation engine that:
- Matches students to jobs
- Detects missing skills
- Suggests learning paths
- Predicts salary and automation risk
- Ranks similar jobs by student fit

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
DATA_DIR = Path("output")
MODEL_DIR = Path("models")
RECO_DIR = Path("recommendation_outputs")
RECO_DIR.mkdir(exist_ok=True)

Load data and models

In [ ]:
jobs = pd.read_csv(DATA_DIR / "jobs_master.csv")
students = pd.read_csv(DATA_DIR / "student_profiles.csv")

salary_pipeline = joblib.load(MODEL_DIR / "salary_pipeline.pkl")
risk_pipeline = joblib.load(MODEL_DIR / "risk_pipeline.pkl")
risk_label_encoder = joblib.load(MODEL_DIR / "risk_label_encoder.pkl")

display(jobs.head())
display(students.head())

,job_title,industry,location,posting_month,job_description,job_description_clean,skills,num_skills,automation_risk_score,automation_risk_label,median_salary_usd
0,data,oowlish technology,remote,2024-10,Oowlish Technology - Founding Engineer - AI/Py...,oowlish technology - founding engineer - ai/py...,"['aws', 'communication', 'git', 'python']",4,44,medium,60000.0
1,other,oneida esc group,remote,2024-10,Oneida Total Integrated Enterprises (OTIE) cur...,oneida total integrated enterprises otie curre...,"['aws', 'excel']",2,52,medium,60000.0
2,management,affinity psychological services. p.c.,remote,2024-10,Affinity Psychological Services is a rapidly-g...,affinity psychological services is a rapidly-g...,"['communication', 'leadership']",2,67,high,60000.0
3,other,the christian and missionary alliance - u.s. c...,remote,2024-10,Job Overview for Director of Children’s Minist...,job overview for director of children s minist...,[],0,55,medium,60000.0
4,data,cdc foundation,full-time,2024-10,CDC Foundation - Data Engineer Data Engineer O...,cdc foundation - data engineer data engineer o...,"['aws', 'azure', 'python', 'r', 'sql']",5,63,medium,60000.0


,course,job_profession,student,linguistic,musical,bodily,logical_mathematical,spatial_visualization,interpersonal,intrapersonal,...,p1,p2,p3,p4,p5,p6,p7,p8,student_id,target_profession
0,NaN,Astronomer\n,S1,11,5,12,16,17,11,18,...,AVG,POOR,AVG,BEST,BEST,AVG,BEST,BEST,student_0,astronomer
1,NaN,Astronomer\n,S2,12,6,12,16,16,11,18,...,AVG,POOR,AVG,BEST,BEST,AVG,BEST,BEST,student_1,astronomer
2,NaN,Astronomer\n,S3,13,7,12,16,15,11,18,...,AVG,POOR,AVG,BEST,BEST,AVG,BEST,BEST,student_2,astronomer
3,NaN,Astronomer\n,S4,14,8,12,16,19,11,18,...,AVG,POOR,AVG,BEST,BEST,AVG,BEST,BEST,student_3,astronomer
4,NaN,Astronomer\n,S5,13,9,12,16,20,11,19,...,AVG,POOR,AVG,BEST,BEST,AVG,BEST,BEST,student_4,astronomer


Clean loaded fields

In [ ]:
jobs["skills"] = jobs["skills"].apply(lambda x: eval(x) if isinstance(x, str) and x.startswith("[") else x)
jobs["skills"] = jobs["skills"].apply(lambda x: x if isinstance(x, list) else [])
jobs["job_description_clean"] = jobs["job_description_clean"].fillna("").astype(str)

##### Define skill ontology

Use the same skill dictionary from notebook 1 so the recommender and training pipeline stay aligned

In [ ]:
SKILL_DICT = {
    "python": ["python"],
    "sql": ["sql", "mysql", "postgresql"],
    "excel": ["excel"],
    "power bi": ["power bi", "powerbi"],
    "tableau": ["tableau"],
    "machine learning": ["machine learning", "ml"],
    "deep learning": ["deep learning"],
    "nlp": ["nlp", "natural language processing"],
    "aws": ["aws"],
    "azure": ["azure"],
    "gcp": ["gcp", "google cloud"],
    "git": ["git", "github"],
    "docker": ["docker"],
    "statistics": ["statistics", "statistical"],
    "communication": ["communication"],
    "leadership": ["leadership"],
    "problem solving": ["problem solving"],
    "prompt engineering": ["prompt engineering", "genai"]
}

##### Student skill extraction

This is the core fix for your earlier issue where current skills and missing skills were not changing. Each student must be converted into a skill profile before matching. A recommendation engine works best when it has a stable student vector and compares it against job requirement vectors

In [ ]:
def extract_student_skills(row):
    skills = []
    text = " ".join([str(v).lower() for v in row.values if pd.notna(v)])
    for canon, pats in SKILL_DICT.items():
        for p in pats:
            if p in text:
                skills.append(canon)
                break
    return sorted(list(set(skills)))

Build student profiles

In [ ]:
student_skill_map = {}

for _, row in students.iterrows():
    sid = row["student_id"]
    student_skill_map[sid] = extract_student_skills(row)

students["skill_profile"] = students["student_id"].map(student_skill_map)
students["skill_profile"] = students["skill_profile"].apply(lambda x: x if isinstance(x, list) else [])

display(students[["student_id", "target_profession", "skill_profile"]].head(10))

,student_id,target_profession,skill_profile
0,student_0,astronomer,[]
1,student_1,astronomer,[]
2,student_2,astronomer,[]
3,student_3,astronomer,[]
4,student_4,astronomer,[]
5,student_5,astronomer,[]
6,student_6,astronomer,[]
7,student_7,astronomer,[]
8,student_8,astronomer,[]
9,student_9,astronomer,[]


Job requirement profiles

In [ ]:
jobs["required_skills"] = jobs["skills"].apply(lambda x: x if isinstance(x, list) else [])
jobs["required_skill_text"] = jobs["required_skills"].apply(" ".join)

display(jobs[["job_title", "location", "industry", "required_skills"]].head(10))

,job_title,location,industry,required_skills
0,data,remote,oowlish technology,"[aws, communication, git, python]"
1,other,remote,oneida esc group,"[aws, excel]"
2,management,remote,affinity psychological services. p.c.,"[communication, leadership]"
3,other,remote,the christian and missionary alliance - u.s. c...,[]
4,data,full-time,cdc foundation,"[aws, azure, python, r, sql]"
5,other,clackamas,worldwide heart to heart ministries,[excel]
6,backend,remote,oowlish technology,"[aws, communication, git, python]"
7,sales,remote,tph corporation,[]
8,other,casa grande,greenlight aba,"[communication, excel]"
9,backend,remote,oowlish technology,"[aws, communication, git, python]"


##### Gap analysis helpers

This section should produce current skills, missing skills, and a learning path. Feature selection and feature overlap are common foundations of career recommendation systems because the model must explain why a job is recommended, not only rank it

In [ ]:
LEARNING_PATH = {
    "python": ["Learn Python basics", "Practice data manipulation with pandas", "Build small analytics projects"],
    "sql": ["Learn SELECT, JOIN, GROUP BY", "Practice joins and subqueries", "Work on analytics SQL problems"],
    "excel": ["Master formulas and pivot tables", "Learn dashboards", "Practice reporting tasks"],
    "power bi": ["Learn data modeling", "Build interactive dashboards", "Connect data sources"],
    "tableau": ["Learn charts and filters", "Build dashboard projects", "Practice storytelling with data"],
    "machine learning": ["Learn supervised learning", "Understand model evaluation", "Build regression/classification projects"],
    "nlp": ["Learn text cleaning", "Understand vectorization", "Build text classification projects"],
    "aws": ["Learn cloud fundamentals", "Study S3 and EC2", "Practice deployment basics"],
    "communication": ["Improve presentation skills", "Practice stakeholder storytelling", "Join communication projects"],
    "leadership": ["Lead small projects", "Practice team coordination", "Take ownership of deliverables"],
    "prompt engineering": ["Learn LLM basics", "Practice prompt design", "Build AI productivity demos"]
}

##### Job matching score
Use a weighted score so results change per student. Do not use identical static match scores. The score should depend on skill overlap, title similarity, location fit, and optionally salary and risk

In [ ]:
def compute_match_score(student_skills, job_skills):
    student_skills = set(student_skills)
    job_skills = set(job_skills)
    if len(job_skills) == 0:
        return 0.0
    return len(student_skills & job_skills) / len(job_skills)

def get_missing_skills(student_skills, job_skills):
    return sorted(list(set(job_skills) - set(student_skills)))

def get_learning_path(missing_skills):
    path = []
    for skill in missing_skills:
        path.extend(LEARNING_PATH.get(skill, [f"Learn {skill}"]))
    return path

##### Prediction helper for each job
Use the trained salary and risk pipelines to produce job-specific predictions. This is where your old bug was happening: the same output repeated because the pipeline was not recomputed per row or the input profile was not changing.

In [ ]:
def predict_job_outcomes(row):
    required_skills = row.get("required_skills", [])
    if isinstance(required_skills, str):
        required_skills = []
    if not isinstance(required_skills, list):
        required_skills = []

    temp = pd.DataFrame([{
        "job_title": str(row.get("job_title", "")).strip().lower(),
        "industry": str(row.get("industry", "")).strip().lower(),
        "location": str(row.get("location", "")).strip().lower(),
        "job_description_clean": str(row.get("job_description_clean", "")).strip().lower(),
        "num_skills": int(len(required_skills)),
        "skill_text": " ".join([str(s).strip().lower() for s in required_skills if pd.notna(s)])
    }])

    for c in temp.columns:
        temp[c] = temp[c].fillna("").astype(str) if c != "num_skills" else pd.to_numeric(temp[c], errors="coerce").fillna(0)

    salary = salary_pipeline.predict(temp)[0]
    salary = float(pd.to_numeric(salary, errors="coerce"))
    if pd.isna(salary):
        salary = 0.0

    risk_idx = risk_pipeline.predict(temp)[0]
    risk_idx = int(pd.to_numeric(risk_idx, errors="coerce"))
    risk = risk_label_encoder.inverse_transform([risk_idx])[0]

    return salary, risk

Personalized recommender

In [ ]:
def recommend_jobs_for_student(student_id, top_n=5):
    student_row = students.loc[students["student_id"] == student_id].iloc[0]
    student_skills = student_row["skill_profile"]
    target = str(student_row.get("target_profession", "")).lower()

    recs = []
    for _, job in jobs.iterrows():
        job_skills = job["required_skills"]
        match_score = compute_match_score(student_skills, job_skills)
        salary_pred, risk_pred = predict_job_outcomes(job)
        missing_skills = get_missing_skills(student_skills, job_skills)

        title_bonus = 0.1 if target and target in str(job["job_title"]).lower() else 0.0
        final_score = 0.7 * match_score + title_bonus

        recs.append({
            "job_title": job["job_title"],
            "location": job["location"],
            "industry": job["industry"],
            "match_score": round(final_score, 4),
            "predicted_salary_usd": round(salary_pred, 2),
            "predicted_automation_risk": risk_pred,
            "missing_skills": ", ".join(missing_skills[:8])
        })

    recs_df = pd.DataFrame(recs).sort_values(
        ["match_score", "predicted_salary_usd"],
        ascending=[False, False]
    ).head(top_n)

    return recs_df

Gap analysis output

In [ ]:
def gap_analysis(student_id, job_title):
    student_row = students.loc[students["student_id"] == student_id].iloc[0]
    job_row = jobs.loc[jobs["job_title"] == job_title].iloc[0]

    current_skills = student_row["skill_profile"]
    required_skills = job_row["required_skills"]
    missing_skills = get_missing_skills(current_skills, required_skills)
    path = get_learning_path(missing_skills)

    return {
        "student_id": student_id,
        "job_title": job_title,
        "current_skills": current_skills,
        "required_skills": required_skills,
        "missing_skills": missing_skills,
        "learning_path": path
    }

In [ ]:
def recommend_jobs_for_student(student_id, top_n=5):
    student_row = students.loc[students["student_id"] == student_id].iloc[0]
    student_skills = student_row["skill_profile"] if isinstance(student_row["skill_profile"], list) else []
    target = str(student_row.get("target_profession", "")).lower()

    recs = []
    for _, job in jobs.iterrows():
        job_skills = job["required_skills"] if isinstance(job["required_skills"], list) else []
        match_score = float(compute_match_score(student_skills, job_skills))
        salary_pred, risk_pred = predict_job_outcomes(job)
        
        salary_pred = pd.to_numeric(salary_pred, errors="coerce")
        if pd.isna(salary_pred):
            salary_pred = 0.0
        
        missing_skills = get_missing_skills(student_skills, job_skills)
        title_bonus = 0.1 if target and target in str(job["job_title"]).lower() else 0.0
        final_score = float(0.7 * match_score + title_bonus)

        recs.append({
            "job_title": str(job["job_title"]),
            "location": str(job["location"]),
            "industry": str(job["industry"]),
            "match_score": final_score,
            "predicted_salary_usd": float(salary_pred),
            "predicted_automation_risk": str(risk_pred),
            "missing_skills": ", ".join(missing_skills[:8])
        })

    recs_df = pd.DataFrame(recs)
    recs_df["match_score"] = pd.to_numeric(recs_df["match_score"], errors="coerce")
    recs_df["predicted_salary_usd"] = pd.to_numeric(recs_df["predicted_salary_usd"], errors="coerce")
    
    recs_df = recs_df.sort_values(
        ["match_score", "predicted_salary_usd"],
        ascending=[False, False]
    ).head(top_n)

    return recs_df

In [ ]:
test_job = jobs.iloc[0]
print(type(test_job.get("job_title")))
print(type(test_job.get("industry")))
print(type(test_job.get("location")))
print(type(test_job.get("required_skills")))

<class 'str'>
<class 'str'>
<class 'str'>
<class 'list'>


In [ ]:
def recommend_jobs_for_student(student_id, top_n=5):
    student_row = students.loc[students["student_id"] == student_id].iloc[0]
    student_skills = student_row["skill_profile"] if isinstance(student_row["skill_profile"], list) else []
    target = str(student_row.get("target_profession", "")).lower().strip()

    recs = []

    for _, job in jobs.iterrows():
        job_skills = job["required_skills"] if isinstance(job.get("required_skills", []), list) else []
        match_score = float(compute_match_score(student_skills, job_skills))
        salary_pred, risk_pred = predict_job_outcomes(job)

        missing_skills = get_missing_skills(student_skills, job_skills)
        title_bonus = 0.1 if target and target in str(job.get("job_title", "")).lower() else 0.0
        final_score = float(match_score * 0.7 + title_bonus)

        recs.append({
            "job_title": str(job.get("job_title", "")),
            "location": str(job.get("location", "")),
            "industry": str(job.get("industry", "")),
            "match_score": final_score,
            "predicted_salary_usd": float(salary_pred),
            "predicted_automation_risk": str(risk_pred),
            "missing_skills": ", ".join(missing_skills[:8])
        })

    recs_df = pd.DataFrame(recs)
    recs_df["match_score"] = pd.to_numeric(recs_df["match_score"], errors="coerce").fillna(0.0)
    recs_df["predicted_salary_usd"] = pd.to_numeric(recs_df["predicted_salary_usd"], errors="coerce").fillna(0.0)

    return recs_df.sort_values(
        by=["match_score", "predicted_salary_usd"],
        ascending=[False, False]
    ).head(top_n)

In [ ]:
sample_student = students["student_id"].iloc[0]
recommendations = recommend_jobs_for_student(sample_student, top_n=10)

recommendations.to_csv(RECO_DIR / "sample_recommendations.csv", index=False)

all_student_recs = []
for sid in students["student_id"].head(20):
    recs = recommend_jobs_for_student(sid, top_n=5)
    recs.insert(0, "student_id", sid)
    all_student_recs.append(recs)

final_recs = pd.concat(all_student_recs, ignore_index=True)
final_recs.to_csv(RECO_DIR / "student_recommendations.csv", index=False)

print("Saved recommendation outputs.")

Saved recommendation outputs.
